# 02 | Data Loading y validación inicial

## Objetivo del notebook

Este notebook carga el dataset crudo y valida que los insumos mínimos estén correctos antes de pasar al EDA y al modelado.

La pregunta central es:

> ¿Tenemos un dataset consistente, con target disponible y columnas suficientes para construir un pipeline reproducible?

No se hacen transformaciones de modelado en este notebook. Solo se inspecciona la fuente.


In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)


Project root: /Users/alexandralozano/dp261-g1-final 2


In [2]:
import pandas as pd
from src.config import RAW_DATA_PATH, TARGET

df = pd.read_csv(RAW_DATA_PATH)
print('Shape:', df.shape)
df.head()


Shape: (72983, 33)


,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,0,1260144000,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,RED,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,0,1260144000,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,WHITE,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,0,1260144000,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,MAROON,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,0,1260144000,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,SILVER,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,0,1260144000,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,SILVER,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020


## 1. Validaciones mínimas

Se revisan tres puntos:

1. Que exista la columna target `IsBadBuy`.
2. Que el target tenga solo valores binarios.
3. Que las columnas del dataset coincidan con el contrato de entrada esperado para API y dashboard.


In [3]:
assert TARGET in df.columns, f'No existe target {TARGET}'
print('Valores del target:', sorted(df[TARGET].dropna().unique()))
print('Columnas:', df.columns.tolist())


Valores del target: [np.int64(0), np.int64(1)]
Columnas: ['IsBadBuy', 'PurchDate', 'Auction', 'VehYear', 'VehicleAge', 'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission', 'WheelTypeID', 'WheelType', 'VehOdo', 'Nationality', 'Size', 'TopThreeAmericanName', 'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice', 'PRIMEUNIT', 'AUCGUART', 'BYRNO', 'VNZIP1', 'VNST', 'VehBCost', 'IsOnlineSale', 'WarrantyCost']


## 2. Tipos de datos crudos

En esta etapa no forzamos tipos definitivos todavía. El pipeline final se encargará de:

- Parsear `PurchDate`.
- Tratar identificadores numéricos como categorías cuando corresponda.
- Convertir precios MMR inválidos en nulos.
- Imputar numéricos y categóricos.
- Codificar categóricos según la familia del modelo.


In [4]:
dtypes = df.dtypes.rename('dtype').reset_index().rename(columns={'index':'column'})
dtypes


,column,dtype
0,IsBadBuy,int64
1,PurchDate,int64
2,Auction,object
3,VehYear,int64
4,VehicleAge,int64
5,Make,object
6,Model,object
7,Trim,object
8,SubModel,object
9,Color,object


## 3. Separación conceptual de variables

El dataset mezcla:

- Variables de vehículo: año, edad, odómetro, marca, modelo, transmisión, color.
- Variables de subasta/mercado: Auction, MMR de adquisición y actuales.
- Variables comerciales: costo de compra, garantía, canal online.
- Identificadores/códigos: BYRNO, VNZIP1, WheelTypeID.

Algunos códigos pueden venir como números, pero no deben tratarse como magnitudes. Por ejemplo, un ZIP mayor no significa más riesgo por ser numéricamente más alto. Por eso el pipeline los convierte a categóricos.


In [5]:
summary = pd.DataFrame({
    'missing': df.isna().sum(),
    'missing_pct': df.isna().mean(),
    'n_unique': df.nunique(dropna=True),
    'dtype': df.dtypes.astype(str),
}).sort_values('missing_pct', ascending=False)
summary.head(20)


,missing,missing_pct,n_unique,dtype
PRIMEUNIT,69564,0.953153,2,object
AUCGUART,69564,0.953153,2,object
WheelType,3174,0.043490,3,object
WheelTypeID,3169,0.043421,4,float64
Trim,2360,0.032336,134,object
MMRCurrentAuctionCleanPrice,315,0.004316,11265,float64
MMRCurrentAuctionAveragePrice,315,0.004316,10315,float64
MMRCurrentRetailCleanPrice,315,0.004316,13192,float64
MMRCurrentRetailAveragePrice,315,0.004316,12493,float64
VehBCost,68,0.000932,2010,float64


## 4. Output esperado

Este notebook no guarda datasets transformados. El guardado de splits se hace en `scripts/run_all.py`, para evitar que diferentes notebooks creen versiones inconsistentes.

El flujo oficial es:

```bash
PYTHONPATH=. python scripts/run_all.py
```
